# 任务1：四层影像组学对照

三种MRI序列与人工标注按MR编号对应。每位患者只使用选定四层，逐层提取特征后汇总为一行，再与年龄、教育年限等临床信息合并。特征提取在保存完整影像的电脑运行；之后可以在这台电脑训练，也可以把特征包传给算力机。


In [ ]:
from pathlib import Path

HERE = Path.cwd().resolve()
CANDIDATES = [HERE, HERE / "1117", HERE.parent / "1117"]
NB_ROOT = next((p for p in CANDIDATES if (p / "config.json").is_file()), None)
if NB_ROOT is None:
    raise FileNotFoundError("请从保存config.json的Notebook目录打开；不需要外层code目录。")


## 实验配置与文件记录

这一组函数负责管理实验文件。`load_config` 读取 `config.json`，`paths` 整理缓存、模型和报告的保存位置；后面的单元通过 `cfg` 读取参数，通过 `p` 查找文件。

`read_cohort` 读取患者清单，`development` 从中取出开发队列。保存结果时，`record` 会同时记录配置和代码版本，便于回查某次结果是怎样得到的。`fresh` 用来检查目标文件是否已存在，重新开展一组实验时可以换一个 `run_id`。


In [ ]:
"""Configuration, provenance and stage boundaries; standard library only."""
from __future__ import annotations

import hashlib
import json
import re
from pathlib import Path

SEED = 20260922
CLINICAL = ["age", "education", "nihss", "hcy", "ua"]
SEQUENCES = ["t1", "t2", "dwi"]
IMAGE_FILES = ["t1_flair.nii.gz", "t2.nii.gz", "dwi_b1000.nii.gz"]


def load_config(path: str | Path) -> dict:
    path = Path(path).resolve()
    cfg = json.loads(path.read_text(encoding="utf-8"))
    settings = path.parent / "experiment_settings.json"
    if settings.is_file():
        overlay = json.loads(settings.read_text(encoding="utf-8"))
        allowed = {"annotation_scope", "optimized", "radiomics", "segmentation", "report_version"}
        if set(overlay) - allowed:
            raise ValueError("experiment_settings.json 只放实验参数，机器路径仍在 config.json。")
        for key, value in overlay.items():
            if isinstance(value, dict):
                cfg[key] = {**cfg.get(key, {}), **value}
            else:
                cfg[key] = value
    if not re.fullmatch(r"[A-Za-z0-9_-]+", cfg["run_id"]):
        raise ValueError("run_id 只使用字母、数字、下划线或短横线。")
    cfg["_root"] = path.parent
    for key in ("excel_path", "patients_root", "existing_cache", "outputs_root"):
        if cfg.get(key):
            p = Path(cfg[key]).expanduser()
            cfg[key] = p if p.is_absolute() else path.parent / p
    cfg["_run"] = cfg["outputs_root"] / cfg["run_id"]
    if cfg.get("data_access") == "classification_cache_only":
        verify_classification_bundle(cfg)
    return cfg




def paths(cfg: dict) -> dict[str, Path]:
    root = cfg["_run"]
    return {
        "run": root, "data": root / "data", "cache": Path(cfg.get("existing_cache") or root / "data" / "resnet50_manual_cache.npz"),
        "adapter": root / "data" / "clinical_adapter.xlsx", "cohort": root / "data" / "cohort.csv",
        "temporal": root / "temporal", "full_cv": root / "full_cv", "single": root / "single_sequence",
        "radiomics": root / "radiomics" / experiment_name(cfg, "radiomics", "selected2d_v1"),
        "explain": root / "explain", "segmentation": root / "segmentation" / experiment_name(cfg, "segmentation", "selected2d_v1"),
        "optimized": root / "optimized" / experiment_name(cfg, "optimized", "resnet_v1"),
        "reports": root / "reports" / safe_name(cfg.get("report_version", "comparison_v2")),
    }




def dump_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str, allow_nan=False), encoding="utf-8")




def fresh(*targets: Path) -> None:
    exists = [str(p) for p in targets if p.exists()]
    if exists:
        raise FileExistsError("产物已存在，请加载已有结果，或更换 run_id：" + ", ".join(exists))




def digest(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()




def source_snapshot(cfg: dict) -> dict:
    """Hash the literal definition cells, independently of outputs and execution counts."""
    result = {}
    for path in sorted(cfg["_root"].glob("*.ipynb")):
        notebook = json.loads(path.read_text(encoding="utf-8"))
        definitions = ["".join(cell["source"]) for cell in notebook["cells"]
                       if cell["cell_type"] == "code" and cell.get("metadata", {}).get("embedded_definition")]
        result[path.name] = hashlib.sha256("\n\n".join(definitions).encode("utf-8")).hexdigest()
    return result




def record(cfg: dict, stage: str, extra: dict | None = None) -> None:
    obj = {"stage": stage, "config": {k: v for k, v in cfg.items() if not k.startswith("_")},
           "notebook_definitions_sha256": source_snapshot(cfg),
           "protocol": "resnet_baseline_and_optimized_selected_slices", **(extra or {})}
    dump_json(paths(cfg)["run"] / "provenance" / f"{stage}.json", obj)




def read_cohort(cfg: dict):
    import pandas as pd
    p = paths(cfg)["cohort"]
    if not p.exists():
        raise FileNotFoundError("先运行00的数据准备与病例清单单元。")
    table = pd.read_csv(p, dtype={"mr": str})
    if not table["mr"].is_unique or not set(table["split"]) <= {"development", "validation"}:
        raise ValueError("病例清单编号重复或队列标记无效。")
    return table




def development(cfg: dict):
    return read_cohort(cfg).query("split == 'development'").reset_index(drop=True)




def require_input(cfg: dict, key: str) -> Path:
    value = cfg.get(key)
    if not value or not Path(value).exists():
        raise FileNotFoundError(f"请在 config.json 配置有效的 {key}。")
    return Path(value)




def verify_classification_bundle(cfg: dict) -> None:
    """Verify portable data without accessing the original image directories."""
    root = cfg["_root"].resolve()
    manifest_path = root / "classification_bundle.json"
    if not manifest_path.is_file():
        raise FileNotFoundError("分类数据包缺少 classification_bundle.json，请上传完整目录。")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("kind") != "classification_cache_only" or manifest["run_id"] != cfg["run_id"]:
        raise ValueError("分类数据包与当前 run_id 不一致。")
    manifest["artifacts"] = {portable_name(k): v for k, v in manifest["artifacts"].items()}
    for key in ("cache", "cohort"):
        manifest[key] = portable_name(manifest[key])
    p = paths(cfg)
    for key in ("cache", "cohort"):
        expected = (root / manifest[key]).resolve()
        expected.relative_to(root)
        if p[key].resolve() != expected or manifest[key] not in manifest["artifacts"]:
            raise ValueError(f"分类数据包的 {key} 路径或清单被改变。")
    for name, expected_hash in manifest["artifacts"].items():
        file = (root / name).resolve()
        file.relative_to(root)
        if not file.is_file() or digest(file) != expected_hash:
            raise ValueError(f"分类数据包文件缺失或校验失败：{name}")


    # Notebook hashes are provenance, not data identity. Model contracts are checked separately.


def require_native_images(cfg: dict) -> None:
    if cfg.get("data_access") == "classification_cache_only":
        raise ValueError("本操作需要原分辨率影像，请在数据电脑执行；导入结果和生成报告无需原影像。")




def safe_name(value):
    if not isinstance(value, str) or not re.fullmatch(r"[A-Za-z0-9_-]+", value):
        raise ValueError("实验名称只使用字母、数字、下划线或短横线。")
    return value




def experiment_name(cfg, stage, default):
    return safe_name(cfg.get(stage, {}).get("experiment", default))




def portable_name(value):
    from pathlib import PurePosixPath
    name = str(value).replace("\\", "/")
    path = PurePosixPath(name)
    if path.is_absolute() or not path.parts or any(p in ("..", ".") or ":" in p for p in path.parts):
        raise ValueError(f"数据包必须使用包内相对路径：{value}")
    return path.as_posix()




def contained_file(root, name):
    root = Path(root).resolve()
    target = (root / portable_name(name)).resolve()
    target.relative_to(root)
    return target




def cohort_identity(cfg):
    """Patient identity independent of local directories, CSV formatting and row order."""
    import pandas as pd
    table = read_cohort(cfg)
    columns = ["mr", "label", "split", "development_fold", "max_layer", "moca"] + CLINICAL
    if not set(columns) <= set(table):
        raise ValueError("病例清单缺少患者、结局、分组、取层或临床列。")
    rows = []
    for _, row in table.sort_values("mr").iterrows():
        values = {}
        for key in columns:
            value = row[key]
            values[key] = (None if pd.isna(value) else str(value) if key in ("mr", "split") else format(float(value), ".12g"))
        rows.append(values)
    payload = {"schema": 1, "patients": rows}
    return hashlib.sha256(json.dumps(payload, sort_keys=True, ensure_ascii=False).encode()).hexdigest()




def definition_contract(cfg, prefix, names):
    import ast
    files = list(cfg["_root"].glob(prefix + "_*.ipynb"))
    if len(files) != 1:
        raise ValueError(f"找不到唯一的 {prefix} Notebook。")
    notebook = json.loads(files[0].read_text(encoding="utf-8"))
    definitions = {}
    for cell in notebook["cells"]:
        if cell["cell_type"] != "code":
            continue
        for node in ast.parse("".join(cell["source"])).body:
            if isinstance(node, (ast.FunctionDef, ast.ClassDef)) and node.name in names:
                if node.name in definitions:
                    raise ValueError(f"重复定义：{node.name}")
                definitions[node.name] = hashlib.sha256(ast.dump(node, include_attributes=False).encode()).hexdigest()
    if set(definitions) != set(names):
        raise ValueError("模型或处理流程定义不齐全。")
    return definitions




def verify_transfer(cfg, directory, kind=None):
    directory = Path(directory).resolve()
    manifest = json.loads((directory / "transfer_manifest.json").read_text(encoding="utf-8"))
    if manifest.get("schema") != 1 or manifest.get("kind") not in ("radiomics_features", "radiomics_complete", "segmentation_results", "segmentation_weights"):
        raise ValueError("不支持的数据包类型或版本。")
    if kind and manifest["kind"] != kind:
        raise ValueError("数据包类型不匹配。")
    if manifest["cohort_identity"] != cohort_identity(cfg):
        raise ValueError("两台机器的患者、标签、临床数据、取层或开发折不一致。")
    if manifest["native_source_identity"] != native_source_identity(cfg):
        raise ValueError("数据包与分类缓存记录的原始影像或人工标注不是同一版本。")
    if manifest["annotation_scope"] != "selected_slices" or cfg["annotation_scope"] != "selected_slices":
        raise ValueError("当前流程只接受四层标注协议。")
    for name, checksum in manifest["artifacts"].items():
        file = contained_file(directory, name)
        if not file.is_file() or digest(file) != checksum:
            raise ValueError(f"数据包文件缺失或传输后发生变化：{name}")
    return manifest




def export_transfer(cfg, source, filenames, destination, kind, protocol):
    import shutil
    source, destination = Path(source).resolve(), Path(destination).resolve()
    fresh(destination)
    mapping = {}
    for name in filenames:
        name = portable_name(name)
        file = contained_file(source, name)
        if not file.is_file():
            raise FileNotFoundError(file)
        mapping[name] = digest(file)
    destination.mkdir(parents=True)
    for name in mapping:
        target = contained_file(destination, name)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(contained_file(source, name), target)
    dump_json(destination / "transfer_manifest.json", {
        "schema": 1, "kind": kind, "cohort_identity": cohort_identity(cfg), "native_source_identity": native_source_identity(cfg),
        "annotation_scope": cfg["annotation_scope"], "protocol": protocol,
        "artifacts": mapping, "source_run_id": cfg["run_id"],
        "notebook_definitions_sha256": source_snapshot(cfg)})
    verify_transfer(cfg, destination, kind)
    return destination




def import_transfer(cfg, source, stage):
    import shutil
    source = Path(source).resolve()
    manifest = verify_transfer(cfg, source)
    allowed = {"radiomics": ("radiomics_features", "radiomics_complete"),
               "segmentation": ("segmentation_results",), "segmentation_weights": ("segmentation_weights",)}
    if stage not in allowed or manifest["kind"] not in allowed[stage]:
        raise ValueError("数据包与接收阶段不匹配。")
    expected_protocol = "selected_four_slices_2d_pixels_v1"
    if manifest["protocol"] != expected_protocol:
        raise ValueError("数据包的二维取层/像素空间处理协议不匹配。")
    target = paths(cfg)[stage] if stage != "segmentation_weights" else paths(cfg)["segmentation"].with_name(paths(cfg)["segmentation"].name + "_imported_model")
    fresh(target)
    target.mkdir(parents=True)
    for name in manifest["artifacts"]:
        dest = contained_file(target, name)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(contained_file(source, name), dest)
    shutil.copy2(source / "transfer_manifest.json", target / "transfer_manifest.json")
    verify_transfer(cfg, target)
    return target




def native_source_identity(cfg):
    audit_file = paths(cfg)["data"] / "volume_audit_details.json"
    audit = json.loads(audit_file.read_text(encoding="utf-8"))
    patients = read_cohort(cfg).mr.tolist()
    files = IMAGE_FILES + ["Untitled.nii.gz"]
    selected = {mr: {name: audit["patients"][mr]["files"][name]["sha256"] for name in files} for mr in patients}
    return hashlib.sha256(json.dumps(selected, sort_keys=True).encode()).hexdigest()




In [ ]:
cfg = load_config(NB_ROOT / "config.json")
p = paths(cfg)
print("实验：", cfg["run_id"])
print("结果目录：", p["run"])


## 读取影像与确认取层

下面的函数检查三序列与标注的尺寸、空间矩阵和取层记录。优先使用患者清单中的路径；换电脑后路径失效时，会在 `patients_root` 下按MR编号寻找唯一目录。四层之外若发现标注，会停止并提示核查。


In [ ]:
"""Native inputs for the confirmed four annotated slices; no unannotated background."""
from pathlib import Path
import json
import re
import numpy as np


def native_patient_files(cfg, patient):
    require_native_images(cfg)
    columns = SEQUENCES + ["manual_mask"]
    recorded = [Path(str(patient.get(c, ""))) for c in columns]
    if all(p.is_file() for p in recorded):
        return recorded
    root = Path(cfg.get("patients_root") or "")
    if not root.is_dir():
        raise FileNotFoundError("在 config.json 填写数据电脑上的 patients_root。")
    if "_native_lookup" not in cfg:
        lookup = {}
        for file in root.rglob(IMAGE_FILES[0]):
            matches = set(re.findall(r"(?<![A-Z0-9])MR\d+(?!\d)", str(file.parent).upper()))
            for mr in matches:
                lookup.setdefault(mr, []).append(file.parent)
        cfg["_native_lookup"] = lookup
    matches = cfg["_native_lookup"].get(str(patient.mr).upper(), [])
    if len(matches) != 1:
        raise ValueError(f"{patient.mr} 未找到唯一的三序列目录；检查 patients_root 和目录中的 MR 编号。")
    files = [matches[0] / name for name in IMAGE_FILES + ["Untitled.nii.gz"]]
    if not all(file.is_file() for file in files):
        raise FileNotFoundError(f"{patient.mr} 缺少三序列影像或人工标注。")
    return files






def selected_indices(patient, depth):
    value = float(patient.max_layer)
    if not np.isfinite(value) or value != int(value):
        raise ValueError(f"{patient.mr} 起始层不是整数。")
    start = int(value) - 1
    zs = list(range(start, start + 4))
    if start < 0 or zs[-1] >= depth:
        raise ValueError(f"{patient.mr} 的四层取层越界。")
    recorded = patient.get("selected_slices_1based")
    if isinstance(recorded, str) and recorded.strip() and json.loads(recorded) != [z + 1 for z in zs]:
        raise ValueError(f"{patient.mr} 的缓存取层与标注层记录不一致。")
    return zs






def selected_patient(cfg, patient):
    import nibabel as nib
    if cfg.get("annotation_scope") != "selected_slices":
        raise ValueError("本流程要求确认全队列仅选定四层标注。")
    files = native_patient_files(cfg, patient)
    volumes = [nib.load(str(file)) for file in files]
    ref = volumes[0]
    tolerance = np.array([1e-4, 1e-4, 1e-4, 1e-3])
    if len(ref.shape) != 3 or any(v.shape != ref.shape or not np.allclose(v.affine, ref.affine, rtol=0, atol=tolerance) for v in volumes[1:]):
        raise ValueError(f"{patient.mr} 的 shape/affine 不一致，先回到00核查。")
    zs = selected_indices(patient, ref.shape[2])
    truth = np.asarray(volumes[-1].dataobj)
    if not np.isfinite(truth).all() or not set(np.unique(truth)) <= {0, 1, 2, 3}:
        raise ValueError(f"{patient.mr} 人工标签必须为0/1/2/3。")
    outside = np.ones(ref.shape[2], dtype=bool)
    outside[zs] = False
    if np.any(truth[:, :, outside] != 0):
        raise ValueError(f"{patient.mr} 在指定四层之外发现标注，请核对起始层。")
    images = [np.asarray(v.dataobj, dtype=np.float32)[:, :, zs] for v in volumes[:3]]
    if any(not np.isfinite(v).all() for v in images):
        raise ValueError(f"{patient.mr} 影像包含非有限值。")
    checksums = {file.name: digest(file) for file in files}
    audit_file = paths(cfg)["data"] / "volume_audit_details.json"
    audit = json.loads(audit_file.read_text(encoding="utf-8"))
    expected = audit["patients"][str(patient.mr)]["files"]
    if any(checksums[name] != expected[name]["sha256"] for name in checksums):
        raise ValueError(f"{patient.mr} 原始影像或标注与00的缓存来源不同，请先核查数据版本。")
    return images, truth[:, :, zs].astype(np.uint8), {
        "mr": str(patient.mr), "slices_1based": [z + 1 for z in zs], "shape": list(ref.shape),
        "native_affine": ref.affine.tolist(), "working_space": "pixel",
        "files_sha256": checksums}






## 计算指标

阈值由开发队列的折外预测确定。时间验证只使用已经确定的阈值，报告AUC、敏感度、特异度和准确率。


In [ ]:
from __future__ import annotations

def threshold_from_development(y, probability):
    fpr, tpr, thresholds = roc_curve(y, probability)
    return float(thresholds[np.argmax(tpr - fpr)])






def classification_metrics(y, probability, threshold, n_bootstrap=1000):
    y, probability = np.asarray(y, dtype=int), np.asarray(probability, dtype=float)
    if len(y) != len(probability) or set(y) != {0, 1} or not np.isfinite(probability).all():
        raise ValueError("标签需包含两类，概率必须有限且与患者一一对应。")
    if ((probability < 0) | (probability > 1)).any():
        raise ValueError("概率不在[0,1]。")
    tn, fp, fn, tp = confusion_matrix(y, probability >= threshold, labels=[0, 1]).ravel()
    result = {"n": len(y), "auc": float(roc_auc_score(y, probability)), "threshold": float(threshold),
              "accuracy": float((tp + tn) / len(y)), "sensitivity": float(tp / (tp + fn)),
              "specificity": float(tn / (tn + fp)), "brier": float(brier_score_loss(y, probability)),
              "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn)}
    if n_bootstrap:
        values = []
        for seed in range(n_bootstrap):
            idx = resample(np.arange(len(y)), replace=True, stratify=y, random_state=seed)
            values.append(roc_auc_score(y[idx], probability[idx]))
        result.update(auc_ci_low=float(np.percentile(values, 2.5)), auc_ci_high=float(np.percentile(values, 97.5)))
    return result






In [ ]:
from sklearn.metrics import brier_score_loss, confusion_matrix, roc_auc_score, roc_curve
from sklearn.utils import resample


## 逐层提取、患者汇总和模型训练

`extract_features` 对每种序列、每个标签逐层提取二维形状、灰度和纹理特征，将有效层的均值、标准差、最大值汇总到患者。空ROI保留缺失，过小且不具备二维范围的ROI只保留数量并记录原因。

这版使用原始平面分辨率、像素单位，不计算三维体积。PyRadiomics逐层做灰度归一化；形状对应像素和像素平方，不能写成mm或mm²。参数与每例影像校验值一起保存。

`fit_radiomics_models` 建立影像组学和影像组学＋五项临床信息两个对照。缺失值填补、标准化、特征筛选及正则强度选择都在训练折内完成。

二维形状实现参考 [PyRadiomics shape2D](https://pyradiomics.readthedocs.io/en/latest/_modules/radiomics/shape2D.html)。


In [ ]:
"""New radiomics baseline, independent of the unchanged ResNet implementation."""
from __future__ import annotations

import json
import numpy as np
import pandas as pd



def extract_features(cfg):
    require_native_images(cfg)
    import SimpleITK as sitk
    import radiomics
    from radiomics import featureextractor
    p, cohort = paths(cfg), read_cohort(cfg)
    destination = p["radiomics"] / "features.csv"
    fresh(destination, p["radiomics"] / "extraction_config.json")
    settings = dict(cfg["radiomics"]["settings"])
    settings.update(force2D=True, force2Ddimension=0, minimumROIDimensions=2)
    if "shape" in cfg["radiomics"]["feature_classes"]:
        raise ValueError("四层标注使用 shape2D，不能使用三维 shape。")
    extractor = featureextractor.RadiomicsFeatureExtractor(**settings)
    extractor.disableAllFeatures()
    for feature_class in cfg["radiomics"]["feature_classes"]:
        extractor.enableFeatureClassByName(feature_class)
    p["radiomics"].mkdir(parents=True, exist_ok=True)
    records, failures, skipped, provenance = [], [], [], []
    for _, patient in cohort.iterrows():
        images, masks, source = selected_patient(cfg, patient)
        provenance.append(source)
        row = {"mr": patient.mr, "label": int(patient.label), "split": patient.split}
        collected = {}
        for label in (1, 2, 3):
            present = [(masks[:, :, index] == label).any() for index in range(4)]
            row[f"roi{label}__present_fraction"] = float(np.mean(present))
            row[f"roi{label}__pixel_count"] = int((masks == label).sum())
            valid_slices = 0
            for index in range(4):
                roi = masks[:, :, index] == label
                coordinates = np.argwhere(roi)
                if not len(coordinates):
                    continue
                if len(coordinates) < 2 or (np.ptp(coordinates, axis=0) < 1).any():
                    skipped.append({"mr": patient.mr, "slice": source["slices_1based"][index],
                                    "label": label, "reason": "ROI不足二维范围；计数保留，纹理记缺失"})
                    continue
                valid_slices += 1
                # Singleton z in SimpleITK order (z, y, x); unit working grid is pixels.
                mask = sitk.GetImageFromArray(masks[:, :, index].T[None].astype(np.uint8))
                mask.SetSpacing((1.0, 1.0, 1.0))
                for channel, sequence in enumerate(SEQUENCES):
                    image = sitk.GetImageFromArray(images[channel][:, :, index].T[None].astype(np.float32))
                    image.CopyInformation(mask)
                    try:
                        features = extractor.execute(image, mask, label=label)
                        for name, value in features.items():
                            if not name.startswith("original_") or ("_shape2D_" in name and sequence != "t1"):
                                continue
                            value = float(value)
                            if np.isfinite(value):
                                collected.setdefault(f"{sequence}__roi{label}__{name}", []).append(value)
                    except (ValueError, RuntimeError) as exc:
                        failures.append({"mr": patient.mr, "slice": source["slices_1based"][index],
                                         "sequence": sequence, "label": label, "error": str(exc)})
            row[f"roi{label}__valid_2d_fraction"] = valid_slices / 4.0
        for feature, values in collected.items():
            for suffix, operation in (("mean", np.mean), ("std", np.std), ("max", np.max)):
                row[f"{feature}__{suffix}"] = float(operation(values))
        records.append(row)
    pd.DataFrame(failures, columns=["mr", "slice", "sequence", "label", "error"]).to_csv(p["radiomics"] / "extraction_errors.csv", index=False)
    pd.DataFrame(skipped, columns=["mr", "slice", "label", "reason"]).to_csv(p["radiomics"] / "small_roi_notes.csv", index=False)
    if failures:
        raise ValueError("提取失败已记录 extraction_errors.csv；处理后再重新提取。")
    table = pd.DataFrame(records)
    table.to_csv(destination, index=False)
    dump_json(p["radiomics"] / "extraction_config.json", {
        "parameters": cfg["radiomics"], "pyradiomics_version": radiomics.__version__,
        "roi_scope": "selected_four_slices_2d_pixels_v1", "cohort_identity": cohort_identity(cfg),
        "features_sha256": digest(destination), "sources": provenance,
        "aggregation": "patient-wise mean/std/max over nonempty valid annotated slices; absent features imputed inside training folds",
        "shape_units": "pixels / pixels squared, not millimetres or whole-lesion volume"})
    record(cfg, "radiomics_features", {"protocol": "selected_four_slices_2d_pixels_v1"})
    return table






def _search(cfg, radiomic_columns, clinical_columns):
    from sklearn.compose import ColumnTransformer
    from sklearn.feature_selection import SelectPercentile, VarianceThreshold, f_classif
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import GridSearchCV, StratifiedKFold
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    rad = Pipeline([("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
                    ("variance", VarianceThreshold()), ("scale", StandardScaler()),
                    ("select", SelectPercentile(f_classif))])
    transformers = [("rad", rad, radiomic_columns)]
    if clinical_columns:
        clinical = Pipeline([("impute", SimpleImputer(strategy="median", keep_empty_features=True)), ("scale", StandardScaler())])
        transformers.append(("clinical", clinical, clinical_columns))
    pipe = Pipeline([("features", ColumnTransformer(transformers)),
                     ("classifier", LogisticRegression(solver="liblinear", max_iter=3000))])
    return GridSearchCV(pipe, {
        "features__rad__select__percentile": cfg["radiomics"]["percentiles"],
        "classifier__C": [0.003, 0.01, 0.03, 0.1, 0.3, 1.0]},
        scoring="roc_auc", cv=StratifiedKFold(5, shuffle=True, random_state=SEED), n_jobs=1, error_score="raise")






def _selected_features(model, columns):
    rad = model.named_steps["features"].named_transformers_["rad"]
    retained = np.asarray(columns)[rad.named_steps["variance"].get_support()]
    return retained[rad.named_steps["select"].get_support()].tolist()






def fit_radiomics_models(cfg):
    import joblib
    p, cohort = paths(cfg), read_cohort(cfg)
    fresh(p["radiomics"] / "models.joblib", p["radiomics"] / "model_meta.json")
    extraction = json.loads((p["radiomics"] / "extraction_config.json").read_text(encoding="utf-8"))
    if extraction["cohort_identity"] != cohort_identity(cfg) or extraction["features_sha256"] != digest(p["radiomics"] / "features.csv"):
        raise ValueError("提取特征后病例清单发生变化。")
    features = pd.read_csv(p["radiomics"] / "features.csv", dtype={"mr": str})
    if not features.mr.is_unique or set(features.mr) != set(cohort.mr):
        raise ValueError("影像组学特征与固定队列不一一对应。")
    matched = features.set_index("mr").loc[cohort.mr]
    if not np.array_equal(matched.label, cohort.label) or matched.split.tolist() != cohort.split.tolist():
        raise ValueError("特征表的标签或时间划分与队列不一致。")
    if extraction["roi_scope"] != "selected_four_slices_2d_pixels_v1":
        raise ValueError("特征提取协议与当前四层二维流程不同。")
    table = cohort.merge(features.drop(columns=["label", "split"]), on="mr", validate="one_to_one")
    table = table.query("split == 'development'").reset_index(drop=True)
    lookup = features.set_index("mr")
    if not np.array_equal(lookup.loc[table.mr, "label"].to_numpy(), table.label.to_numpy()):
        raise ValueError("影像组学结局与队列不一致。")
    radiomic_columns = [c for c in features if c not in ("mr", "label", "split")]
    if not radiomic_columns:
        raise ValueError("没有影像组学特征。")
    y = table.label.to_numpy(dtype=int)
    fitted, metadata, predictions, selection_rows = {}, {}, [], []
    for name, clinical_columns in (("Radiomics", []), ("Radiomics_Clinical5", CLINICAL)):
        columns = radiomic_columns + clinical_columns
        X = table[columns].replace([np.inf, -np.inf], np.nan)
        oof, fold_auc = np.zeros(len(y), dtype=float), []
        for fold in range(1, 6):
            valid = table.development_fold.to_numpy() == fold
            search = _search(cfg, radiomic_columns, clinical_columns)
            search.fit(X.loc[~valid], y[~valid])
            oof[valid] = search.predict_proba(X.loc[valid])[:, 1]
            from sklearn.metrics import roc_auc_score
            fold_auc.append(float(roc_auc_score(y[valid], oof[valid])))
            for feature in _selected_features(search.best_estimator_, radiomic_columns):
                selection_rows.append({"model": name, "fold": fold, "feature": feature})
        search = _search(cfg, radiomic_columns, clinical_columns)
        search.fit(X, y)
        fitted[name] = search.best_estimator_
        threshold = threshold_from_development(y, oof)
        if not np.isfinite(threshold):
            raise ValueError(f"{name}未形成有限Youden阈值；先检查开发预测。")
        metadata[name] = {"features": columns, "threshold": threshold, "best_params": search.best_params_,
                          "selected_features": _selected_features(search.best_estimator_, radiomic_columns),
                          "mean_fold_auc": float(np.mean(fold_auc)), "fold_aucs": fold_auc,
                          "development": classification_metrics(y, oof, threshold, n_bootstrap=0)}
        prediction = table[["mr", "label", "development_fold"]].copy()
        prediction["model"], prediction["probability"] = name, oof
        predictions.append(prediction)
    joblib.dump(fitted, p["radiomics"] / "models.joblib")
    pd.concat(predictions).to_csv(p["radiomics"] / "development_oof.csv", index=False)
    pd.DataFrame(selection_rows).to_csv(p["radiomics"] / "selected_features_by_fold.csv", index=False)
    dump_json(p["radiomics"] / "model_meta.json", metadata)
    dump_json(p["radiomics"] / "frozen_artifacts.json", {"artifacts": {
        filename: digest(p["radiomics"] / filename) for filename in ("models.joblib", "model_meta.json", "features.csv", "extraction_config.json")},
        "cohort_identity": cohort_identity(cfg)})
    record(cfg, "radiomics_models", {"feature_selection": "inside_inner_cv"})
    return metadata






def predict_radiomics_validation(cfg):
    p, cohort = paths(cfg), read_cohort(cfg)
    receipt = p["radiomics"] / "transfer_manifest.json"
    if receipt.exists():
        manifest = verify_transfer(cfg, p["radiomics"])
        if manifest["kind"] == "radiomics_complete":
            return imported_radiomics_predictions(cfg)
    import joblib
    lock = json.loads((p["radiomics"] / "frozen_artifacts.json").read_text(encoding="utf-8"))
    if lock["cohort_identity"] != cohort_identity(cfg):
        raise ValueError("固定队列发生变化。")
    for name, expected in lock["artifacts"].items():
        if digest(p["radiomics"] / name) != expected:
            raise ValueError(f"影像组学固定产物发生变化：{name}")
    feature_table = pd.read_csv(p["radiomics"] / "features.csv", dtype={"mr": str})
    val = cohort.query("split == 'validation'").merge(feature_table.drop(columns=["label", "split"]), on="mr", validate="one_to_one")
    models = joblib.load(p["radiomics"] / "models.joblib")
    meta = json.loads((p["radiomics"] / "model_meta.json").read_text(encoding="utf-8"))
    predictions, summaries = [], []
    for name, model in models.items():
        prob = model.predict_proba(val[meta[name]["features"]].replace([np.inf, -np.inf], np.nan))[:, 1]
        threshold = meta[name]["threshold"]
        table = val[["mr", "label"]].copy()
        table["model"], table["probability"], table["threshold"] = name, prob, threshold
        table["prediction"] = (prob >= threshold).astype(int)
        table["protocol"] = "radiomics_selected_four_slices_nested_feature_selection"
        predictions.append(table)
        summaries.append({"model": name, **classification_metrics(val.label, prob, threshold)})
    return pd.concat(predictions, ignore_index=True), pd.DataFrame(summaries)






def imported_radiomics_predictions(cfg):
    root = paths(cfg)["radiomics"]
    verify_transfer(cfg, root, "radiomics_complete")
    table = pd.read_csv(root / "validation_predictions.csv", dtype={"mr": str})
    expected = read_cohort(cfg).query("split == 'validation'").set_index("mr")
    meta = json.loads((root / "model_meta.json").read_text(encoding="utf-8"))
    if set(table.model) != {"Radiomics", "Radiomics_Clinical5"}:
        raise ValueError("影像组学预测缺少对照组。")
    summaries = []
    for name, group in table.groupby("model"):
        if not group.mr.is_unique or set(group.mr) != set(expected.index):
            raise ValueError("导入预测的时间验证患者不一致。")
        if not np.array_equal(group.label, expected.loc[group.mr, "label"]):
            raise ValueError("导入预测结局不一致。")
        threshold = float(meta[name]["threshold"])
        if not np.allclose(group.threshold, threshold, rtol=0, atol=1e-12):
            raise ValueError("导入预测阈值与开发阶段不同。")
        if not np.array_equal(group.prediction, (group.probability >= threshold).astype(int)):
            raise ValueError("导入预测分类与概率不一致。")
        summaries.append({"model": name, **classification_metrics(group.label, group.probability, threshold)})
    return table, pd.DataFrame(summaries)






def export_radiomics(cfg, destination, include_models=True):
    root = paths(cfg)["radiomics"]
    extraction = json.loads((root / "extraction_config.json").read_text(encoding="utf-8"))
    if extraction["cohort_identity"] != cohort_identity(cfg) or extraction["features_sha256"] != digest(root / "features.csv"):
        raise ValueError("特征表或队列在提取后发生变化。")
    files = ["features.csv", "extraction_config.json", "extraction_errors.csv", "small_roi_notes.csv"]
    kind = "radiomics_features"
    if include_models:
        predictions, _ = predict_radiomics_validation(cfg)
        target = root / "validation_predictions.csv"
        if target.exists():
            existing = pd.read_csv(target, dtype={"mr": str})
            pd.testing.assert_frame_equal(existing, predictions, check_dtype=False, rtol=1e-10, atol=1e-12)
        else:
            predictions.to_csv(target, index=False)
        files += ["models.joblib", "model_meta.json", "frozen_artifacts.json", "development_oof.csv",
                  "selected_features_by_fold.csv", "validation_predictions.csv"]
        kind = "radiomics_complete"
    return export_transfer(cfg, root, files, destination, kind, "selected_four_slices_2d_pixels_v1")






## 1. 提取特征（数据电脑）

确认 `config.json` 的 `patients_root` 指向完整数据。`experiment_settings.json` 已记录全队列四层标注规则。旧清单中的“unconfirmed”是确认前的记录，不要求为此重做分类缓存。


In [ ]:
RUN_EXTRACTION = False
if RUN_EXTRACTION:
    features = extract_features(cfg)
    display(features.head())


## 2. 导入特征包或完整结果包（算力机，可选）

把数据电脑导出的整个文件夹上传后，修改下面的路径。导入检查患者编号、标签、五项临床信息、取层、开发折和文件内容；原电脑的盘符不参与一致性判断。目标实验目录已经存在时不会覆盖。


In [ ]:
RUN_IMPORT_RADIOMICS = False
RADIOMICS_PACKAGE = NB_ROOT / "incoming" / "radiomics_complete"
if RUN_IMPORT_RADIOMICS:
    print(import_transfer(cfg, RADIOMICS_PACKAGE, "radiomics"))


## 3. 训练两个对照模型

数据电脑提取完特征后即可执行；算力机导入“仅特征包”后也可以执行。导入完整结果包时跳过此步，直接在06统一比较。


In [ ]:
RUN_RADIOMICS_FIT = False
if RUN_RADIOMICS_FIT:
    display(fit_radiomics_models(cfg))


## 4. 导出到算力机

`EXPORT_WITH_MODELS=False` 导出特征表和处理记录，用于异地训练。设为 `True` 时，还会导出两组模型、开发OOF和时间验证预测；06可以直接读取预测，不必重新安装影像读取库或在异地加载模型。完整包的导出应放在开发方案确定之后，不能根据时间验证表现反复挑方案。


In [ ]:
RUN_EXPORT_RADIOMICS = False
EXPORT_WITH_MODELS = True
RADIOMICS_EXPORT = NB_ROOT / "transfer_packages" / "radiomics_complete"
if RUN_EXPORT_RADIOMICS:
    print(export_radiomics(cfg, RADIOMICS_EXPORT, include_models=EXPORT_WITH_MODELS))


## 查看已保存的结果

特征表中的一行对应一位患者；模型摘要中的开发指标来自折外预测。优先比较同一队列上的pooled AUC，另列五折均值及波动。


In [ ]:
for filename in ("features.csv", "development_oof.csv", "selected_features_by_fold.csv"):
    file = p["radiomics"] / filename
    if file.exists():
        print(filename)
        display(pd.read_csv(file).head())
